<div align="center">
    
<h1> BetFlow: Blockchain Analytics and Forensics Pipeline </h1>
<p> Profilazione Comportamentale, Ricostruzione Topologica delle Sessioni e De-anonimizzazione OSINT sul Protocollo SatoshiDice </p>

<br>

<h2>Samuele Vasta</h2>

<br>

</div>

---

### Abstract
Il presente documento analitico illustra l'architettura ingegneristica progettata per tracciare, ricostruire e modellare le dinamiche transazionali generate da **SatoshiDice**, la piattaforma prototipale ad alta densità operante sulla blockchain di Bitcoin. 

Attraverso l'applicazione congiunta della *Teoria dei Grafi*, dell'inferenza statistica e di tecniche avanzate di memory optimization, il sistema estrae e profila pattern comportamentali esogeni, mappando le latenze di payload e i cluster deterministici su un ledger storico che cattura ogni singola transazione generata dal Genesis Block (3 Gennaio 2009) al blocco `214.562` (31 Dicembre 2012).

## 1. Architettura di Ingestion e Ottimizzazione della Memoria

L'ingestione massiva di dati da un ledger distribuito richiede un controllo rigoroso delle risorse in virtù dell'elevata dimensionalità. Per ottimizzare l'allocazione della RAM e prevenire saturazioni di memoria (*Out-of-Memory*), il caricamento dei dataset è interamente gestito dal modulo `src.data_loader`.

Il processo automatizzato implementa le seguenti ottimizzazioni di performance:
- **Algorithmic Downcasting**: Forzatura proattiva dei tipi di dato inferiti da standard generici (es. `int64`, 64-bit) a tipi compatti asimmetrici o minimi strettamente necessari (es. `uint32`, interi a 32-bit senza segno, e booleani a 8-bit). Questa manipolazione garantisce una **compressione strutturale del footprint di RAM del 50%**, rendendo elaborabili volumi massivi in memoria.
- **Micro-Allocazione per Hash (PyArrow)**: Adozione del tipo vettoriale `string[pyarrow]` per ottimizzare radicalmente lo stoccaggio e la ricerca degli hash crittografici SHA-256 a lunghezza fissa.
- **I/O Multi-Threading**: Sostituzione del parser C standard della libreria Pandas con l'engine `pyarrow`, operante in multi-threading, riducendo drasticamente le latenze I/O per i set di dati transazionali.

In [1]:
import sys
import os

# Aggiungiamo la root del progetto al sys.path per permettere a Python di trovare 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src import load_all_data

tx, inputs, outputs, mapping, satoshi_dice = load_all_data()

'transactions.csv' loaded successfully. Size: 211.74 MB
'inputs.csv' loaded successfully. Size: 244.66 MB
'outputs.csv' loaded successfully. Size: 469.47 MB
'mapAddr2Ids8708820.csv' loaded successfully. Size: 381.54 MB
'satoshiDiceInfos.tsv' loaded successfully. Size: 0.0029 MB


## 2. Analisi generali delle bet e dei payout

In questa macro-sezione si intende esplorare l'impatto di SatoshiDice sull'ecosistema Bitcoin, misurando l'entità dei flussi economici e definendo il comportamento temporale e transazionale dell'utenza. Verranno evasi i punti specifici delineati nei requisiti.

### 2.1 Transazioni Bet vs Rete Globale (Percentuale nel tempo)
**Obiettivo:** Calcolare la percentuale di transazioni di bet rispetto al numero complessivo di transazioni, in relazione a diversi periodi temporali. Verificheremo il grado di congestione causato da SatoshiDice sulla blockchain in intervalli di tempo predefiniti (es. mensili).

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: Raggruppamento per mese del volume transazioni Satoshi Dice vs Rete globale
# ... (implementazione analisi) ...


### 2.2 Popolarità degli Indirizzi (Frequenza e Volumi)
**Obiettivo:** Valutare la popolarità di ogni indirizzo di betting con prefisso `1dice`, computando sia il numero assoluto di puntate ricevute sia il volume economico mosso in BTC (ammontare complessivo). I risultati verranno presentati attraverso visualizzazioni grafiche comparative.

In [3]:
# TODO: Filtraggio degli address '1dice', conteggio invocazioni e calcolo volumi BTC
# ... (implementazione analisi) ...


### 2.3 Distanza tra Transazioni di Bet e correlato Payout
**Obiettivo:** Misurare la latenza ("distanza") tra una scommessa e l'emissione del payout ad essa associato. L'associazione si verifica quando la transazione di payout originata dal servizio consuma esattamente un UTXO (output non speso) della relativa transazione di bet. Questa metrica verrà espressa considerando il delta temporale o la differenza in block heights.

In [ ]:
# TODO: Corrispondenza degli UTXO tra input delle tx di payout e output delle bet
# Calcolo della latenza e distribuzione delle distanze (es. Block Height Delta o Time Delta)
# ... (implementazione analisi latenza) ...

### 2.4 Profilazione comportamentale (Top 3 Indirizzi Popolari)
**Obiettivo:** Condurre un deep-dive specifico sui tre indirizzi `1dice` più utilizzati per intercettare regolarità o pattern particolari. Saranno indagate tre dimensionalità:
1. **Distribuzione temporale delle bet**. Ripetizione di puntate suddivise per arco temporale predefinito (per ora, giorno, o settimana).
2. **Correlazione Fee-Amount**. Individuazione di legami statistici (tramite scatter plot o similar) tra i miner fee pagati e il volume scommesso.
3. **Intervallo tra le bet consecutive**. Analisi della distribuzione dei tempi di stop tra le puntate dello stesso cluster/user (per rilevare comportamenti costanti dettati da potenziali bot, o attività organica).

In [ ]:
# TODO: Individuazione della Top 3 Addresses
# - Grafici per le distribuzioni temporali delle bet
# - Correlazione (Scatter Plot) della fee allocata contro l'amount della bet
# - Distribuzione tempi d'attesa (deltas) tra bet consecutive dello stesso path/utente
# ... (implementazione analisi Top 3 addresses) ...

## 3. Analisi di sequenze di bet (Graph Reconstruction)

In questa sezione verranno ripristinate le catene logiche di interazione con il protocollo. Al fine di mantenere alta la confidenza analitica, il pool si restringe unicamente alle **"Simple Bets"** relative al singolo indirizzo amministrativo `1dice` dominante nel layer prelevato.

**Workflow Analitico:**
1. Isolamento delle Simple Bets (Tx configurate rigidamente su: 1 Input univoco $\rightarrow$ 2 Output [Destinazione 1dice, Resto *Change*]).
2. Generazione del **Grafo Diretto $G = (V,E)$**. I nodi sono le transazioni, e un arco connette il nodo $v_i$ al nodo $v_j$ se e solo se l'input della tx $v_j$ assorbe direttamente l'output *Change* di scarto della tx $v_i$. L'etichetta dell'arco codifica esplicitamente il *Change Address*.
3. Individuazione delle componenti lineari massimali e ricostruzione formale dei longest-paths (Sessioni Utente).
4. Rappresentazione grafica della distribuzione di lunghezza di tali catene (su unità di nodi e/o archi) per inferire la persistenza della base comportamentale giocante.

In [ ]:
import networkx as nx

# TODO: Costruzione del Grafo Diretto (G) seguendo la struttura Input/Change-Output.
# - Filtraggio Simple Bets target
# - Isolamento delle sequenze (Paths persistenti) e calcolo delle lunghezze
# - Plotting della distribuzione di lunghezza delle sub-catene
# ... (implementazione network analysis) ...

## 4. Scraping di WalletExplorer: De-anonimizzazione OSINT

La fase terminale affronta la riconciliazione e il clustering di entità, spostandosi dalla topologia nativa ad intelligence esogena (**WalletExplorer**).

L'auditing si restringe esplicitamente e specificatamente alle **Top $h$ catene di lunghezza massima individuate nel Grafo**. L'approccio sfrutterà strategie di *Heuristic Clustering* interrogando l'open source tramite agenti d'automazione del browser (`selenium`). 

Per ciascun longest path, l'ispezione estrarrà iterativamente l'appartenenza strutturale del wallet. L'esito finale consoliderà un dataset (Comprehensive Evaluation Table) dotato dei seguenti *mandatory fields*:
- Identificativo logico univoco della Catena.
- Lunghezza parametrizzata (Archi / Nodi).
- Volume asimmetrico totale degli address unici attivati all'interno dello stream.
- Densità vettoriale dei Wallet identificati (e relativo indice di concentrazione percentuale al predominante).
- *Identificativo Tag* esplicito del Wallet predominante de-anonimizzato o ID cluster se anonimo.

In [ ]:
# TODO: Implementazione di Selenium webdriver headless
# - Estrazione Change Addresses per le top H catene 
# - Scraping dinamico delle associazioni e Wallet da WalletExplorer
# - Generazione della tabella riassuntiva dei riscontri
# ... (implementazione WalletExplorer scraping pipeline) ...